<a href="https://colab.research.google.com/github/athitthiyan/Learning_Gen_AI/blob/main/Day4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:


# RUN THIS CELL TO SETUP THE CHALLENGE DATA
import pandas as pd
import sqlite3



conn_challenge = sqlite3.connect(':memory:')



challenge_data = {
    "Visit_ID": [5001, 5001, 5002, 5003],
    "Student_ID": [101, 101, 102, 104],
    "Student_Name": ["Alice", "Alice", "Bob", "David"],
    "Doctor_ID": ["DOC_XYZ", "DOC_XYZ", "DOC_ABC", "DOC_XYZ"],
    "Doctor_Name": ["Dr. Evans", "Dr. Evans", "Dr. Green", "Dr. Evans"],
    "Doctor_Clinic": ["General Medicine", "General Medicine", "Sports Med", "General Medicine"],
    "Prescriptions": ["Amoxicillin, Ibuprofen", "Amoxicillin, Ibuprofen", "Bandages", "Vitamin D"]
}



df_challenge = pd.DataFrame(challenge_data)
df_challenge.to_sql('Patient_Visits_0NF', conn_challenge, index=False, if_exists='replace')



print("--- Challenge Dataset (0NF) ---")
df_challenge

--- Challenge Dataset (0NF) ---


,Visit_ID,Student_ID,Student_Name,Doctor_ID,Doctor_Name,Doctor_Clinic,Prescriptions
0,5001,101,Alice,DOC_XYZ,Dr. Evans,General Medicine,"Amoxicillin, Ibuprofen"
1,5001,101,Alice,DOC_XYZ,Dr. Evans,General Medicine,"Amoxicillin, Ibuprofen"
2,5002,102,Bob,DOC_ABC,Dr. Green,Sports Med,Bandages
3,5003,104,David,DOC_XYZ,Dr. Evans,General Medicine,Vitamin D


In [5]:
# Flattening the Prescriptions column to ensure atomicity

df_1nf = df_challenge.assign(Prescriptions=df_challenge['Prescriptions'].str.split(', ')).explode('Prescriptions')



# Save to SQL

df_1nf.to_sql('Table_1NF', conn_challenge, index=False, if_exists='replace')



print("--- 1NF Data (Atomic values enforced) ---")

print(f"Row count increased from {len(df_challenge)} to {len(df_1nf)} due to flattening.")

df_1nf

--- 1NF Data (Atomic values enforced) ---
Row count increased from 4 to 6 due to flattening.


,Visit_ID,Student_ID,Student_Name,Doctor_ID,Doctor_Name,Doctor_Clinic,Prescriptions
0,5001,101,Alice,DOC_XYZ,Dr. Evans,General Medicine,Amoxicillin
0,5001,101,Alice,DOC_XYZ,Dr. Evans,General Medicine,Ibuprofen
1,5001,101,Alice,DOC_XYZ,Dr. Evans,General Medicine,Amoxicillin
1,5001,101,Alice,DOC_XYZ,Dr. Evans,General Medicine,Ibuprofen
2,5002,102,Bob,DOC_ABC,Dr. Green,Sports Med,Bandages
3,5003,104,David,DOC_XYZ,Dr. Evans,General Medicine,Vitamin D


In [8]:
# Create Students table
df_students_2nf = df_1nf[['Student_ID', 'Student_Name']].drop_duplicates().reset_index(drop=True)

# Create Courses table
df_doctors_2nf = df_1nf[
    ['Doctor_ID', 'Doctor_Name', 'Doctor_Clinic']
].drop_duplicates().reset_index(drop=True)

# Create Enrollment table
df_visits_2nf = df_1nf[
    ['Visit_ID', 'Student_ID', 'Doctor_ID', 'Prescriptions']
].drop_duplicates().reset_index(drop=True)

# Save tables into SQL
df_students_2nf.to_sql('Students_2NF', conn_challenge, index=False, if_exists='replace')
df_doctors_2nf.to_sql('Doctors_2NF', conn_challenge, index=False, if_exists='replace')
df_visits_2nf.to_sql('Visits_2NF', conn_challenge, index=False, if_exists='replace')

print("\n--- 2NF Tables Created ---")

print("\n[Students_2NF]")
display(df_students_2nf)

print("\n[Doctors_2NF]")
display(df_doctors_2nf)

print("\n[Visits_2NF]")
display(df_visits_2nf)


--- 2NF Tables Created ---

[Students_2NF]


,Student_ID,Student_Name
0,101,Alice
1,102,Bob
2,104,David



[Doctors_2NF]


,Doctor_ID,Doctor_Name,Doctor_Clinic
0,DOC_XYZ,Dr. Evans,General Medicine
1,DOC_ABC,Dr. Green,Sports Med



[Visits_2NF]


,Visit_ID,Student_ID,Doctor_ID,Prescriptions
0,5001,101,DOC_XYZ,Amoxicillin
1,5001,101,DOC_XYZ,Ibuprofen
2,5002,102,DOC_ABC,Bandages
3,5003,104,DOC_XYZ,Vitamin D


In [9]:
# Extract Doctors into a separate table to eliminate transitive dependency

df_doctors_3nf = df_doctors_2nf[
    ['Doctor_ID', 'Doctor_Name', 'Doctor_Clinic']
].drop_duplicates().reset_index(drop=True)

# Clean up Visits table to keep only Doctor_ID as a Foreign Key

df_visits_3nf = df_visits_2nf[
    ['Visit_ID', 'Student_ID', 'Doctor_ID', 'Prescriptions']
].drop_duplicates().reset_index(drop=True)

# Save to SQL

df_doctors_3nf.to_sql(
    'Doctors_3NF',
    conn_challenge,
    index=False,
    if_exists='replace'
)

df_visits_3nf.to_sql(
    'Visits_3NF',
    conn_challenge,
    index=False,
    if_exists='replace'
)

print("--- 3NF Schema Refinement ---")

print("\n[Visits_3NF] (No transitive dependency):")
display(df_visits_3nf)

print("\n[Doctors_3NF] (New isolated table):")
display(df_doctors_3nf)

--- 3NF Schema Refinement ---

[Visits_3NF] (No transitive dependency):


,Visit_ID,Student_ID,Doctor_ID,Prescriptions
0,5001,101,DOC_XYZ,Amoxicillin
1,5001,101,DOC_XYZ,Ibuprofen
2,5002,102,DOC_ABC,Bandages
3,5003,104,DOC_XYZ,Vitamin D



[Doctors_3NF] (New isolated table):


,Doctor_ID,Doctor_Name,Doctor_Clinic
0,DOC_XYZ,Dr. Evans,General Medicine
1,DOC_ABC,Dr. Green,Sports Med
